In [1]:
import pandas as pd

import numpy as np

## **Business case**

We are data analysts at an international company that sells prosthetics to hospitals, and the Advertising department wants to run a marketing campaign targeting people who have suffered shark attacks. They have asked us to determine which age group, country, and type of incidents the campaign should focus on.

## **Data used**

##### - **GSAF file**: Compiles shark attacks that have occurred historically worldwide.

In [2]:
df = pd.read_csv('GSAF5.csv', sep=';')

df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source
0,11th October,2025,Unprovoked,Australia,Queensland,Cook Esplanade Thursday Island,Fishing/swimming,Samuel Nai,M,14,Serious abdonminal injuries,N,1823 hrs,Tiger or Bull shark,Kevin McMurray Trackingsharks.com
1,7th October,2025,Unprovoked,Australia,South Australia,Kangaroo Island,Surfing,Lee Berryman,M,50+,Lacerations to calf,N,1330hrs,Bronze whaler?,Kevin McMurray Trackingsharks.com
2,29th September,2025,Unprovoked,USA,Off California,Catalina Island,Swimming,Christopher Murray,M,54,Leg and foot injury,N,0100hrs,unknown 1.2m shark,Todd Smith: Kevin McMurray Trackingsharks.com
3,27th September,2025,Provoked,Costa Rica,NaN,Cocos Islands,Diving-Tagging sharks,Dr. Mauricio Hoyos,M,48,Head face and arms,N,Not stated,Tiger shark 4m,Todd Smith: Kevin McMurray Trackingsharks.com
4,6th September,2025,Unprovoked,Australia,NSW,Long Reef Sydney,Surfing,Mercury Psillaskis,M,57,Both legs and arm severed,Y,0930hrs,Great White Shark,Todd Smith: Andy Currie: Simon De Marchi: Kevi...


##### - **Trade file**: Compiles money invested by country in human prostheses.

In [3]:
trade_data = pd.read_csv('TradeData.csv', sep=';')

# Renaming columns for clarity
trade_data = trade_data.rename(columns={'reporterDesc_cleaned': 'Country', 'Total_Trade_Value_Formatted': 'Invested Value'})

# Simplifying trade dataframe with relevant columns
trade_data = trade_data[['Country', 'Invested Value']]

trade_data.head()

,Country,Invested Value
0,United States,"$379,980,041,869"
1,Switzerland,"$265,215,193,004"
2,Germany,"$204,475,690,572"
3,United Kingdom,"$179,828,485,414"
4,Netherlands,"$177,121,565,695"


## **Data cleaning**

##### **GSAF file**

##### - Activity column

In [4]:
# Filling empty rows with unknown activity
df['Activity'] = df['Activity'].fillna("Unknown (not reported)")

# Mapping various activities to broader categories
activity_mapping = {
    'dive': 'Snorkeling/Diving',
    'diving': 'Snorkeling/Diving',
    'surf': 'Surfing/Paddle surfing or related',
    'swim': 'Swimming/Bathing',
    'fish': 'Fishing or related',
    'wading': 'Walking/standing/doing something beside the beach',
    'boat': 'Boating/Sailing',
    'kayak': 'Kayaking or related',
    'snorkel': 'Snorkeling/Diving',
    'yacht': 'Boating/Sailing',
    'walk': 'Walking/standing/doing something beside the beach',
    'paddl': 'Surfing/Paddle surfing or related',
    'board': 'Surfing/Paddle surfing or related',
    'canoe': 'Kayaking or related',
    'sail': 'Boating/Sailing',
    'bath': 'Swimming/Bathing',
    'float': 'Swimming/Bathing',
    'standing': 'Walking/standing/doing something beside the beach',
    'tread': 'Swimming/Bathing',
    'dangl': 'Swimming/Bathing',
    'rowing': 'Kayaking or related',
    'disaster': 'Natural/Circunstancial disasters',
    'wreck': 'Natural/Circunstancial disasters',
    'murder': 'Natural/Circunstancial disasters',
    'netting': 'Fishing or related',
    'splash': 'Swimming/Bathing',
    'play': 'Swimming/Bathing',
    'jump': 'Swimming/Bathing',
    'fell': 'Swimming/Bathing',
    'ski': 'Surfing/Paddle surfing or related',
    'shark': 'Sharks interaction',
    'launch': 'Boating/Sailing',
    'adrift': 'Natural/Circunstancial disasters',
    'ship': 'Boating/Sailing',
    'life': 'Lifeguarding/Rescuing',
    'rescu': 'Lifeguarding/Rescuing',
    'sup': 'Surfing/Paddle surfing or related',
    'aircraft': 'Natural/Circunstancial disasters',
    'craft': 'Boating/Sailing',
    'catch': 'Fishing or related',
    'crab': 'Fishing or related',
    'wash': 'Walking/standing/doing something beside the beach',
    'torpedo': 'Natural/Circunstancial disasters',
    'clam': 'Fishing or related',
    'collect': 'Fishing or related',
    'sit': 'Walking/standing/doing something beside the beach',
    'scul': 'Swimming/Bathing',
    'unknown': 'Unknown (not reported)',
    'suicide': 'Natural/Circunstancial disasters',
}

# Creating a function to go through each activity row
def categorize(activity):
    text = str(activity).lower()
    for key, value in activity_mapping.items():
        if key in text:
            return value
    return 'Other (not classified)'

# Applying new categories to activities
df['Activity group'] = df['Activity'].apply(categorize).fillna("Unknown (not reported)")

##### - Country column

In [5]:
# Clean and standardise the 'Country' column
df['Country'] = df['Country'].astype(str).str.strip().str.title()
# Define corrections and ambiguous entries
corrections = {
    'Usa': 'United States',
    'Columbia': 'Colombia',
    'Maldive Islands': 'Maldives',
    'Turks & Caicos': 'Turks And Caicos',
    'Trinidad & Tobago': 'Trinidad And Tobago',
    'St. Martin': 'Saint Martin',
    'St Martin': 'Saint Martin',
    'St. Maartin': 'Saint Martin',
    'St Kitts / Nevis': 'Saint Kitts And Nevis',
    'Nevis': 'Saint Kitts And Nevis',
    'Canary Islands': 'Spain',
    'Hawaii': 'United States',
    'Puerto Rico': 'United States',
    'Guam': 'United States',
    'American Samoa': 'United States',
    'Johnston Island': 'United States',
    'New Britain': 'Papua New Guinea',
    'New Guinea': 'Papua New Guinea',
    'Roatan': 'Honduras',
    'San Domingo': 'Dominican Republic',
    'Ceylon': 'Sri Lanka',
    'Ceylon (Sri Lanka)': 'Sri Lanka'
}
ambiguous = [
    'Asia?', 'Ocean', 'Indian Ocean?', 'Red Sea?', 'British Overseas Territory',
    'British West Indies', 'British Isles', 'Africa', 'Coast Of Africa',
    'The Balkans', 'Nan', 'Mid-Pacifc Ocean', 'Red Sea / Indian Ocean',
    'Between Portugal & India', 'Equatorial Guinea / Cameroon',
    'Andaman / Nicobar Islandas', 'South Pacific Ocean', 'South Atlantic Ocean',
    'North Atlantic Ocean', 'North Pacific Ocean', 'South China Sea',
    'Mediterranean Sea', 'Caribbean Sea', 'Atlantic Ocean', 'Pacific Ocean',
    'Mid Atlantic Ocean', 'Northern Arabian Sea', 'Central Pacific',
    'Bay Of Bengal', 'Tasman Sea', 'North Sea', 'Persian Gulf', 'Coral Sea',
    'Diego Garcia', 'Admiralty Islands', 'West Indies'
]
# Apply corrections
df['Country'] = df['Country'].replace(corrections)
# Replace ambiguous entries with NaN and clean unique country names
df['Country'] = df['Country'].replace(ambiguous, np.nan).dropna()

##### - Fatal Y/N column

In [6]:
# Asegurarnos de que todos los valores sean texto, eliminar cualquier espacio en blanco y convertirlos a mayúsculas.
column_name = 'Fatal Y/N'
# 1. Convertir a string (para manejar NaN y 2017)
# 2. Eliminar espacios en blanco alrededor
# 3. Convertir a mayúsculas
df[column_name] = df[column_name].astype(str).str.strip().str.upper()
# Damos por hecho que los valores NQ e Y x 2 son erratas a la hora de escribir y que corresponden a N e Y respectivamente
df[column_name] = df[column_name].replace({'Y X 2': 'Y', 'NQ': 'N'})
# Examinar los valores que quedan y que no son 'Y' o 'N'. Estos son los valores que hay que convertir a np.nan porque son inconsistentes o ambiguos.
column_name = 'Fatal Y/N'
nan_values = ['NAN', 'F', 'M', 'UNKNOWN', '2017']
# Reemplazar estos valores por np.nan
df[column_name] = df[column_name].replace(nan_values, np.nan)

##### - Age column

In [7]:
def clean_age(age):
    if pd.isna(age):  # If the value is missing
        return np.nan
    age = str(age).strip().lower()  # Convert to string, remove spaces, make lowercase
    # Remove words that are not actual ages
    if age in ['young', 'adult', 'elderly', 'teen', 'teens', 'a minor', 'middle age', 'make line green', 'x', 'f', 'm', 'a.m.']:
        return np.nan
    # Remove entries with symbols or multiple ages
    if any(char in age for char in ['&', ',', '/', '-', 'to', '?']):
        return np.nan
    # Try to convert to an integer
    try:
        return int(float(age))  # Convert to float first, then to int
    except:
        return np.nan
# Apply the function to your DataFrame
df['Age_cleaned'] = df['Age'].apply(clean_age)

##### - Injury column

In [8]:
# 1. Definir la columna
column_name = 'Injury'
# 2. Convertir a string
# 3. Eliminar espacios en blanco
# 4. Convertir a mayúsculas
df[column_name] = df[column_name].astype(str).str.strip().str.upper()
# Mostramos los 10 valores más comunes para verificar la estandarización
def categorize_injury_refined(injury: str) -> str:
    """
    Redefinir las categorías por tipo (Extremidades vs. Tronco).
    """
    # 1. Manejo de valores faltantes/desconocidos
    if pd.isna(injury) or injury == 'NO DETAILS' or injury == 'UNKNOWN':
        return 'DESCONOCIDO'
    injury = injury.upper()
    # Identificación del tipo de lesión
    # Palabras clave para Extremidades Inferiores (Piernas, Pies, Tobillos, etc.)
    LOWER_EXTREMITY = ('LEG', 'FOOT', 'ANKLE', 'CALF', 'THIGH', 'KNEE', 'HEEL')
    # Palabras clave para Extremidades Superiores (Brazos, Manos, Muñecas, etc.)
    UPPER_EXTREMITY = ('ARM', 'HAND', 'WRIST', 'FINGER', 'ELBOW')
    # Palabras clave para Tronco/Cabeza/General (Torso, Cabeza, Espalda, etc.)
    TRUNK_HEAD = ('HEAD', 'FACE', 'NECK', 'SHOULDER', 'TORSO', 'BACK', 'CHEST', 'ABDOMEN')
    is_lower = any(part in injury for part in LOWER_EXTREMITY)
    is_upper = any(part in injury for part in UPPER_EXTREMITY)
    is_trunk = any(part in injury for part in TRUNK_HEAD)
    # 1. Casos de Fatalidad
    if 'FATAL' in injury or 'DIED' in injury or 'DEATH' in injury:
        return 'FATAL'
    # 2. Casos de NO Lesión
    if 'NO INJURY' in injury or 'NO INJURY TO OCCUPANTS' in injury or 'SURVIVED' in injury:
        return 'NO LESION / SOLO EQUIPO'
    # 3. Lesiones Graves
    if 'SEVERED' in injury or 'AMPUTATED' in injury or 'LOST' in injury or 'BIT OFF' in injury:
        if is_lower:
            return 'GRAVE - EXTREMIDAD INFERIOR'
        if is_upper:
            return 'GRAVE - EXTREMIDAD SUPERIOR'
        return 'GRAVE - OTRA ZONA'
    # 4. Lesiones Menores/Mordedura
    if 'LACERATED' in injury or 'LACERATIONS' in injury or 'PUNCTURE' in injury or 'BITTEN' in injury or 'INJURED' in injury or 'MINOR' in injury:
        if is_lower:
            return 'MENOR - EXTREMIDAD INFERIOR'
        if is_upper:
            return 'MENOR - EXTREMIDAD SUPERIOR'
        if is_trunk:
            return 'MENOR - TRONCO/CABEZA'
        return 'MENOR - OTRA ZONA' # Para casos como "Minor injury" sin especificar parte
    # 5. Otros
    return 'OTRO'
# Aplicar la función y crear la nueva columna
column_name = 'Injury'
# Aplicar la función de categorización refinada
df['Injury_Category_Refined'] = df[column_name].apply(categorize_injury_refined)
# Mostramos el conteo de la nueva columna

##### - Attack count column

In [9]:
df['Attack Count'] = 1
df['Attack Count'] = df['Attack Count'].astype(int)

##### **Trade file**

##### - Invested value column

In [10]:
# Converting Invested Value to numeric and filling Nans with 0
trade_data.loc[:, 'Invested Value'] = trade_data['Invested Value'].str.replace('$', '').str.replace(',', '').fillna(0).astype('float64')

# Ensuring all columns are of the correct type
trade_data['Invested Value'] = trade_data['Invested Value'].astype('float64')

##### - Invested value in million column

In [11]:
# Creating Invested Value in Million column to have clearer data and filling Nans with 0
trade_data.loc[:, 'Invested Value Million'] = trade_data['Invested Value'] / 1_000_000
trade_data.loc[:, 'Invested Value Million'] = trade_data['Invested Value Million'].fillna(0)

# Ensuring all columns are of the correct type
trade_data['Invested Value Million'] = trade_data['Invested Value Million'].astype('float64')

## **Data analysis**

##### - Grouping attacks count by country

In [12]:
country_attacks = df.groupby('Country')['Attack Count'].agg(['sum']).reset_index().rename(columns={'sum': 'Total Attacks'})
country_attacks.head()

,Country,Total Attacks
0,Algeria,1
1,Andaman Islands,1
2,Angola,1
3,Antigua,2
4,Argentina,2


##### - Merging country attacks with trade data

In [13]:
country_trade = pd.merge(country_attacks, trade_data, on='Country', how='left').rename(columns={'sum': 'Total Attacks'}).sort_values(by='Total Attacks', ascending=False)
country_trade.head()

,Country,Total Attacks,Invested Value,Invested Value Million
152,United States,2585,3.799800e+11,379980.041869
6,Australia,1510,3.710396e+10,37103.958180
128,South Africa,599,6.275345e+09,6275.345366
106,Papua New Guinea,152,3.246075e+06,3.246075
97,New Zealand,146,3.113536e+09,3113.535796


##### - Studying correlation between variables 'Total Attacks' and 'Invested Value Million'

In [14]:
country_trade['Total Attacks'].corr(country_trade['Invested Value Million'])

np.float64(0.7433477763390549)

##### **Conclusion**

There is a strong positive correlation between investment in prosthetics (in millions) and the number of shark attacks per country. This suggests that countries with a higher number of attacks also tend to invest more in prosthetics, likely as a consequence of the increased incidents themselves.

##### - Studying how much is invested per shark attack in each country

In [15]:
country_trade['Invest per Attack'] = (country_trade['Invested Value Million'] / country_trade['Total Attacks'].replace(0, np.nan))
country_candidates = country_trade[country_trade['Total Attacks'] > 50].sort_values(by='Invest per Attack', ascending=True)
country_candidates.head(5)

,Country,Total Attacks,Invested Value,Invested Value Million,Invest per Attack
106,Papua New Guinea,152,3.246075e+06,3.246075,0.021356
41,Fiji,70,4.082108e+06,4.082108,0.058316
8,Bahamas,141,1.303439e+07,13.034391,0.092442
128,South Africa,599,6.275345e+09,6275.345366,10.476370
35,Egypt,53,1.077981e+09,1077.981295,20.339270


##### **Conclusion**
The countries with the greatest business opportunities are Papua New Guinea, the Bahamas, South Africa, New Zealand, and Australia, as they are the countries with more than 50 recorded attacks and the lowest investment per attack, according to the data.


##### - Studying if there is a pattern of what caused the attacks in selected countries for product designing

In [16]:
# Filter the DataFrame with the top 5 countries from country_candidates
df = df[df['Country'].isin(country_candidates['Country'].head(5))]

# Defining the pivot table comparing Activity groups and Countries
activity_attack_country = df.pivot_table(
    index='Activity group',
    columns='Country',
    values='Attack Count',
    aggfunc='sum',
    fill_value=0
)

# Converting to percentages within each country
activity_attack_country = activity_attack_country.div(activity_attack_country.sum(axis=0), axis=1).round(2) * 100

activity_attack_country

Country,Bahamas,Egypt,Fiji,Papua New Guinea,South Africa
Activity group,,,,,
Boating/Sailing,2.0,0.0,3.0,1.0,1.0
Fishing or related,38.0,6.0,33.0,41.0,23.0
Kayaking or related,0.0,0.0,3.0,2.0,0.0
Lifeguarding/Rescuing,0.0,0.0,0.0,0.0,1.0
Natural/Circunstancial disasters,3.0,4.0,0.0,1.0,1.0
Other (not classified),1.0,2.0,1.0,5.0,1.0
Sharks interaction,5.0,2.0,0.0,0.0,3.0
Snorkeling/Diving,26.0,49.0,23.0,11.0,7.0
Surfing/Paddle surfing or related,5.0,2.0,4.0,1.0,29.0


##### **Conclusion**
There is no a common pattern of what has caused the attacks in each country, so marketing campaigns/product design should be different in each country depending on the activity caused.


##### - Studying the amount of attacks caused in each country to prepare available stock depending on sizes

In [17]:
df['Age_cleaned'] = df['Age_cleaned'].fillna(0).astype('int64')

df['Age group'] = pd.cut(
    df['Age_cleaned'],
    bins=[-1, 0, 12, 19, 35, 50, 65, 100],
    labels=[
        'Unknown',
        'Child (0-12)',
        'Teen (13-19)',
        'Young Adult (20-35)',
        'Adult (36-50)',
        'Middle Aged (51-65)',
        'Senior (66+)'
    ]
)

# Defining the pivot table comparing Age groups and Countries
age_attack_country = df.pivot_table(
    index='Age group',
    columns='Country',
    values='Attack Count',
    aggfunc='sum',
    fill_value=0
)

# Converting to percentages within each country
age_attack_country = age_attack_country.div(age_attack_country.sum(axis=0), axis=1).round(2) * 100

age_attack_country

C:\Users\gcast\AppData\Local\Temp\ipykernel_2140\3598791435.py:18: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  age_attack_country = df.pivot_table(


Country,Bahamas,Egypt,Fiji,Papua New Guinea,South Africa
Age group,,,,,
Unknown,36.0,57.0,50.0,66.0,38.0
Child (0-12),4.0,2.0,1.0,5.0,1.0
Teen (13-19),7.0,8.0,6.0,12.0,19.0
Young Adult (20-35),21.0,9.0,31.0,15.0,32.0
Adult (36-50),20.0,11.0,10.0,1.0,8.0
Middle Aged (51-65),11.0,4.0,0.0,0.0,2.0
Senior (66+),2.0,9.0,1.0,0.0,1.0


##### **Conclusion**
There is not enough info to calculate the amount of product of each size is needed, as almost half of the ages in each country are unknown.
